In [ ]:
import pandas as pd
import numpy as np
import re
import pickle
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Bidirectional, Dropout

# 1. دالة التنظيف (التي أصلحنا فيها الـ NameError)
def advanced_clean(text):
    if not isinstance(text, str):
        return ""
    # إزالة الروابط
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    # إزالة التشكيل
    tashkeel = re.compile(r'[\u0617-\u061A\u064B-\u0652]')
    text = re.sub(tashkeel, '', text)
    # إزالة الرموز والأرقام والحفاظ على الحروف العربية
    text = re.sub(r'[^\u0600-\u06ff\s]', ' ', text)
    text = re.sub(r'\d+', ' ', text)
    # توحيد المسافات
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# 2. تنظيف البيانات (حذف الفارغ والمكرر)
print("جاري تنظيف البيانات...")
df.dropna(subset=['text', 'label'], inplace=True)
df.drop_duplicates(subset=['text'], inplace=True)
df['text'] = df['text'].apply(advanced_clean)
df = df[df['text'].str.strip() != ""]

# 3. موازنة البيانات (7000 نص لكل فئة)
print("جاري موازنة الفئات...")
target_number = 7000
df_balanced = df.groupby('label').apply(
    lambda x: x.sample(n=min(len(x), target_number), random_state=42)
).reset_index(drop=True)

# 4. تحويل النصوص إلى أرقام (Tokenizer & Padding)
le = LabelEncoder()
y = le.fit_transform(df_balanced['label'])

max_words = 15000
max_len = 150
tokenizer = Tokenizer(num_words=max_words, oov_token="<OOV>")
tokenizer.fit_on_texts(df_balanced['text'])

X = pad_sequences(tokenizer.texts_to_sequences(df_balanced['text']), maxlen=max_len, padding='post', truncating='post')
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 5. بناء هيكل الـ RNN (Bidirectional LSTM)
model = Sequential([
    Embedding(max_words, 64, input_length=max_len),
    Bidirectional(LSTM(64, return_sequences=True)),
    Dropout(0.2),
    Bidirectional(LSTM(32)),
    Dense(32, activation='relu'),
    Dense(len(le.classes_), activation='softmax')
])

model.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

# 6. التدريب (5 Epochs)
print("بدء عملية التدريب...")
history = model.fit(X_train, y_train, epochs=5, batch_size=64, validation_data=(X_test, y_test))

# 7. الحفظ النهائي
model.save('arabic_rnn_model.h5')
with open('tokenizer.pickle', 'wb') as handle:
    pickle.dump(tokenizer, handle)
with open('label_encoder.pickle', 'wb') as handle:
    pickle.dump(le, handle)

print("✅ تمت المهمة بنجاح!")

In [8]:
import tensorflow as tf
import pickle
import numpy as np
import re
from tensorflow.keras.preprocessing.sequence import pad_sequences

# 1. تحديد المسارات الصحيحة (مباشرة بدون join)
model_path = r'C:\Users\Owner\Desktop\model\RNN\arabic_rnn_model.h5'
tokenizer_path = r'C:\Users\Owner\Desktop\model\RNN\tokenizer.pickle'
le_path = r'C:\Users\Owner\Desktop\model\RNN\label_encoder.pickle'

# 2. تحميل الملفات
try:
    model = tf.keras.models.load_model(model_path)
    with open(tokenizer_path, 'rb') as handle:
        tokenizer = pickle.load(handle)
    with open(le_path, 'rb') as handle:
        le = pickle.load(handle)
    print("✅ تم تحميل النموذج والملحقات بنجاح!")
except Exception as e:
    print(f"❌ خطأ في التحميل: {e}")

# 3. دالة التنظيف
def clean_text(text):
    text = re.sub(r'http\S+|www\S+|https\S+', '', str(text))
    tashkeel = re.compile(r'[\u0617-\u061A\u064B-\u0652]')
    text = re.sub(tashkeel, '', text)
    text = re.sub(r'[^\u0600-\u06ff\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# 4. دالة للتنبؤ بمجموعة نصوص دفعة واحدة
def predict_bulk(texts_list):
    # تنظيف كل النصوص في القائمة
    cleaned_texts = [clean_text(t) for t in texts_list]
    
    # تحويل النصوص إلى تسلسلات أرقام
    sequences = tokenizer.texts_to_sequences(cleaned_texts)
    padded = pad_sequences(sequences, maxlen=150, padding='post')
    
    # التنبؤ (دفعة واحدة أسرع من التنبؤ الفردي)
    predictions = model.predict(padded, verbose=0)
    
    results = []
    for pred in predictions:
        label_idx = np.argmax(pred)
        results.append(le.classes_[label_idx])
    return results

# --- تجربة إدخال عدة نصوص ---
my_texts = [
    "المنتخب الوطني يحقق فوزاً ثميناً في تصفيات كأس العالم",
    "انخفاض أسعار الذهب في الأسواق العالمية اليوم",
    "دراسة جديدة تؤكد فوائد النوم المبكر على صحة القلب",
    "وزير الخارجية يبحث سبل التعاون الثنائي مع نظيره الفرنسي"
]

predictions = predict_bulk(my_texts)

print("\n--- نتائج التصنيف ---")
for txt, label in zip(my_texts, predictions):
    print(f"النص: {txt}")
    print(f"التصنيف: {label}")
    print("-" * 20)

✅ تم تحميل النموذج والملحقات بنجاح!

--- نتائج التصنيف ---
النص: المنتخب الوطني يحقق فوزاً ثميناً في تصفيات كأس العالم
التصنيف: Sport
--------------------
النص: انخفاض أسعار الذهب في الأسواق العالمية اليوم
التصنيف: Economy
--------------------
النص: دراسة جديدة تؤكد فوائد النوم المبكر على صحة القلب
التصنيف: Society
--------------------
النص: وزير الخارجية يبحث سبل التعاون الثنائي مع نظيره الفرنسي
التصنيف: Politic
--------------------
